# ST-OMR Meter V3-A3 Residual Calibration Screen

REAL-only deterministic screen. No D10 access, no optimizer, no weight update, TEST sealed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, torch

REPO_URL = "https://github.com/khfy7wpr5p-maker/st-omr-training.git"
REPO_REF = "fix/meter-real-domain-adaptation-v3-a3-residual-calibration"
WORK_ROOT = Path("/content/st-omr-meter-v3-a3-screen")
REPO_DIR = WORK_ROOT / "repo"

PARENT_RUN_ROOT = Path(
    "/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS/"
    "meter-real-domain-v3-a2-positive-margin-run"
)
PARENT_RESUME = PARENT_RUN_ROOT / "resume.pt"

PILOT_ROOT = Path(
    "/content/drive/MyDrive/TEST/METER_V1/00_AUDIT/"
    "teacher_gold_pilot_v1"
)
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS/"
    "meter-real-domain-v3-a3-residual-calibration-screen"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

subprocess.run(
    ["git", "clone", "--branch", REPO_REF, "--single-branch", "--filter=blob:none",
     REPO_URL, str(REPO_DIR)],
    check=True,
)

repository_sha = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if not PARENT_RESUME.is_file():
    raise FileNotFoundError(PARENT_RESUME)

print(json.dumps({
    "experiment": "meter-real-domain-adaptation-v3-a3-residual-calibration-screen",
    "repository_sha": repository_sha,
    "parent_resume": str(PARENT_RESUME),
    "d10_opened": False,
    "optimizer_steps": 0,
    "test_opened": False,
}, indent=2))


In [ ]:
from st_omr_training.meter_real_domain_adaptation_v3_a3 import (
    calibrated_logits_v3_a3,
    classification_summary_v3_a3,
    gain_pairs_milli_v3_a3,
    real_phase0_gate_v3_a3,
    select_gain_pair_v3_a3,
    verify_parent_resume_metadata_v3_a3,
)
from st_omr_training.meter_real_domain_adaptation_v3_a2 import (
    build_meter_classification_adapter_v3_a2,
)
from st_omr_training.meter_real_domain_adaptation_v1 import (
    _load_teacher_records,
    _stack_teacher,
)
from st_omr_training.meter_teacher_gold_admission_v1 import (
    build_meter_teacher_gold_bundle_v1,
    verify_meter_teacher_gold_bundle_v1,
)
from st_omr_training.stage7d11_barline_meter_training import (
    build_meter_refiner,
    FROZEN_D11_CONFIG,
)

snapshot = torch.load(PARENT_RESUME, map_location="cpu", weights_only=True)
verify_parent_resume_metadata_v3_a3(snapshot)

base_model = build_meter_refiner(FROZEN_D11_CONFIG)
parent_model = build_meter_classification_adapter_v3_a2(base_model)
parent_model.load_state_dict(snapshot["current_model_state"], strict=True)

for parameter in parent_model.parameters():
    parameter.requires_grad = False
parent_model.eval()

if any(parameter.requires_grad for parameter in parent_model.parameters()):
    raise RuntimeError("FAIL-CLOSED: V3-A3 parent must be fully frozen")

teacher_candidates = [
    Path("/content/st-omr-meter-v3-a2-sparse/teacher-gold-bundle-v1"),
    Path("/content/st-omr-meter-v3-a1-sparse/teacher-gold-bundle-v1"),
]
TEACHER_ROOT = next(
    (p for p in teacher_candidates if p.is_dir() and (p / "COMPLETE").is_file()),
    None,
)

if TEACHER_ROOT is None:
    TEACHER_ROOT = WORK_ROOT / "teacher-gold-bundle-v1"
    build_meter_teacher_gold_bundle_v1(
        pilot_path=PILOT_ROOT / "pilot-data.json",
        choices_path=PILOT_ROOT / "ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json",
        permission_evidence_path=PILOT_ROOT / "meter-training-permission-evidence-v1.json",
        privacy_review_evidence_path=PILOT_ROOT / "meter-privacy-review-evidence-v1.json",
        output_root=TEACHER_ROOT,
        repository_root=REPO_DIR,
    )
else:
    verify_meter_teacher_gold_bundle_v1(TEACHER_ROOT)

records = _load_teacher_records(TEACHER_ROOT)
train_records = tuple(record for record in records if record.split == "train")
validation_records = tuple(record for record in records if record.split == "validation")

if len(train_records) != 54 or len(validation_records) != 18:
    raise RuntimeError("FAIL-CLOSED: Teacher Gold split cardinality changed")

print(json.dumps({
    "parent_adaptation_version": snapshot["adaptation_version"],
    "parent_repository_sha": snapshot["repository_sha"],
    "completed_epoch": int(snapshot["completed_epoch"]),
    "best_epoch": int(snapshot["best_epoch"]),
    "teacher_train": len(train_records),
    "teacher_validation": len(validation_records),
    "parent_fully_frozen": True,
}, indent=2))


In [ ]:
def cached_components(model, selected_records):
    images, classes, _boxes, _positive = _stack_teacher(selected_records)
    with torch.no_grad():
        logits, _boxes2, _presence, _digits, base_logits, adapter_logits = model.components(images)
    return classes, logits, base_logits, adapter_logits

train_classes, parent_train_logits, train_base_logits, train_adapter_logits = cached_components(
    parent_model, train_records
)

parent_train_summary = classification_summary_v3_a3(
    train_classes.tolist(),
    parent_train_logits.argmax(1).tolist(),
)

candidate_train_summaries = {}
for gain_2_4_milli, gain_4_4_milli in gain_pairs_milli_v3_a3():
    candidate_logits = calibrated_logits_v3_a3(
        train_base_logits,
        train_adapter_logits,
        gain_2_4_milli=gain_2_4_milli,
        gain_4_4_milli=gain_4_4_milli,
    )
    candidate_train_summaries[(gain_2_4_milli, gain_4_4_milli)] = classification_summary_v3_a3(
        train_classes.tolist(),
        candidate_logits.argmax(1).tolist(),
    )

selection = select_gain_pair_v3_a3(
    parent_train_summary=parent_train_summary,
    candidate_train_summaries=candidate_train_summaries,
)

validation_classes, parent_validation_logits, validation_base_logits, validation_adapter_logits = cached_components(
    parent_model, validation_records
)

parent_validation_summary = classification_summary_v3_a3(
    validation_classes.tolist(),
    parent_validation_logits.argmax(1).tolist(),
)

calibrated_validation_logits = calibrated_logits_v3_a3(
    validation_base_logits,
    validation_adapter_logits,
    gain_2_4_milli=selection.gain_2_4_milli,
    gain_4_4_milli=selection.gain_4_4_milli,
)

calibrated_validation_summary = classification_summary_v3_a3(
    validation_classes.tolist(),
    calibrated_validation_logits.argmax(1).tolist(),
)

phase0_accepted, phase0_reasons = real_phase0_gate_v3_a3(calibrated_validation_summary)

def summary_payload(summary):
    return {
        "record_count": summary.record_count,
        "macro_f1": summary.macro_f1,
        "accuracy": summary.accuracy,
        "per_class_recall": dict(summary.per_class_recall),
        "confusion": [list(row) for row in summary.confusion],
    }

result = {
    "schema": "st-omr-meter-v3-a3-residual-calibration-screen-v1",
    "repository_sha": repository_sha,
    "parent_resume_sha256": __import__("hashlib").sha256(PARENT_RESUME.read_bytes()).hexdigest(),
    "parent_repository_sha": snapshot["repository_sha"],
    "parent_adaptation_version": snapshot["adaptation_version"],
    "grid": {
        "gain_2_4_min": 1.0,
        "gain_2_4_max": 1.25,
        "gain_4_4_min": 1.0,
        "gain_4_4_max": 1.25,
        "step": 0.025,
        "candidate_pairs": 121,
        "gain_none": 1.0,
        "gain_3_4": 1.0,
    },
    "selected_from_real_train_only": {
        "gain_2_4": selection.gain_2_4_milli / 1000.0,
        "gain_4_4": selection.gain_4_4_milli / 1000.0,
    },
    "parent_real_train": summary_payload(parent_train_summary),
    "selected_real_train": summary_payload(selection.train_summary),
    "parent_real_validation": summary_payload(parent_validation_summary),
    "calibrated_real_validation": summary_payload(calibrated_validation_summary),
    "phase0_gate": {
        "accepted": phase0_accepted,
        "reasons": list(phase0_reasons),
    },
    "d10_opened": False,
    "optimizer_steps": 0,
    "weights_modified": False,
    "bbox_frozen_exact": True,
    "test_opened": False,
    "runtime_connected": False,
    "production_promotion_authorized": False,
}

out = OUTPUT_ROOT / "phase0-real-screen.json"
out.write_text(
    json.dumps(result, sort_keys=True, separators=(",", ":"), ensure_ascii=True),
    encoding="ascii",
)

print("==============================================")
print("V3-A3 SELECTED GAINS")
print("==============================================")
print(json.dumps(result["selected_from_real_train_only"], indent=2))

print("\n==============================================")
print("PARENT REAL VALIDATION")
print("==============================================")
print(json.dumps(result["parent_real_validation"], indent=2))

print("\n==============================================")
print("CALIBRATED REAL VALIDATION")
print("==============================================")
print(json.dumps(result["calibrated_real_validation"], indent=2))

print("\n==============================================")
print("PHASE 0 GATE")
print("==============================================")
print(json.dumps(result["phase0_gate"], indent=2))

print("\n==============================================")
print("SAFETY")
print("==============================================")
print(json.dumps({
    "d10_opened": False,
    "optimizer_steps": 0,
    "weights_modified": False,
    "bbox_frozen_exact": True,
    "test_opened": False,
}, indent=2))

print("\nRESULT:", out)
